# Stage C — Final

The shipped Stage C: a batched **`validate_rct`** LLM judge on **`yandexgpt-5-lite`** (probe winner) at batch 20, dropping funnel false positives. Clean `stage_c_filter(sentences)` + a demo + the SAL headline.

## What Stage C is

The funnel's **Stage A** (section-aware localize) + **Stage B** (cue rules + frozen head) is
**recall-first and cheap**, but it leaks false positives: prior-work critiques ("existing methods
cannot…"), contribution claims ("we propose…"), acknowledgments, formulas. **Stage C** asks an LLM,
per surviving candidate, *"is this really the authors' own gap?"* and drops the No's.

Because A+B already cut ~98% of sentences, the LLM sees only ~6 sentences/paper — so LLM-grade
precision costs ~cents per thousands of papers, not the ~$4000/1M of sending whole papers to an LLM.

- Implementation: **`LLMGapFilter`** (imported from `gap2idea`), batched judge, strict-JSON output.
- Prompt mode **`validate_rct`** (biomedical, keep-biased) for the SAL/RCT domain below.
- Backend: **Yandex paid models** (they honor strict `json_schema` — the free tier does not).

> **Key principle:** Stage C is a *precision* filter. It only **improves F1 where there are true
> negatives to remove** — so we evaluate on **SAL** (has negatives), not on positives-only gold
> (where a filter can only cost recall).

In [ ]:
# ===== Stage C setup: Yandex paid models via the repo's LLM layer =====
import os, sys, time, csv, io, json, urllib.request, random
from pathlib import Path
import numpy as np, pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "gap2idea").exists())
sys.path.insert(0, str(ROOT / "src")); sys.path.insert(0, str(ROOT / "scripts" / "bench"))
from dotenv import load_dotenv; load_dotenv(ROOT / ".env")
os.environ["LLM_PROVIDER"] = "yandex"                       # use Yandex paid models

from gap2idea.pipeline.gap_llm_filter import LLMGapFilter, SYSTEM_VALIDATE_RCT, BATCH_CHUNK
from gap2idea.pipeline.llm import get_llm_client, active_provider

FOLDER = os.environ["YANDEX_FOLDER_ID"].strip()
def yuri(name): return f"gpt://{FOLDER}/{name}/latest"     # full URI passes through the client untouched
CLIENT = get_llm_client("yandex")
print("provider:", active_provider(), "| default batch chunk:", BATCH_CHUNK)

In [ ]:
# Final Stage C config
WINNER = yuri("yandexgpt-5-lite")   # probe winner; yuri("yandexgpt-5-pro") for robustness
CHUNK  = 20
MODE   = "validate_rct"             # biomedical; use "validate" for general / arXiv text

def stage_c_filter(sentences, model=WINNER, chunk=CHUNK, mode=MODE):
    """Keep only sentences the LLM confirms are the authors' OWN gap. Batched; returns kept list."""
    f = LLMGapFilter(backend="api", model=model, mode=mode, client=CLIENT)
    keep=[]
    for k in range(0, len(sentences), chunk):
        keep.extend(f.judge_batch(sentences[k:k+chunk]))
    return [s for s,kp in zip(sentences,keep) if kp], f.n_calls

demo = [
    "A limitation of our trial is the small sample size and short follow-up.",
    "The study was single-center, which may limit generalizability.",
    "Existing methods cannot capture long-range dependencies.",   # prior work -> drop
    "We thank the reviewers for their comments.",                 # ack -> drop
    "Future work will evaluate the approach in a multi-center setting.",
    "Our method achieves state-of-the-art accuracy.",             # boast -> drop
]
kept, calls = stage_c_filter(demo)
print(f"kept {len(kept)}/{len(demo)} in {calls} call(s):")
for s in kept: print("  KEEP:", s)

## Headline on SAL

In [ ]:
# Headline SAL number (prevalence-matched vs Lan 2024 PubMedBERT 0.821)
from bench_rct import load as sal_load
from bench_limgen import prf
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
TEST_CAP = 1500   # None for the full stream

tr_s, tr_y = sal_load("train"); te_s, te_y = sal_load("test", TEST_CAP)
enc = SentenceTransformer("BAAI/bge-small-en-v1.5")
emb = lambda xs: enc.encode(xs, normalize_embeddings=True, batch_size=128, show_progress_bar=False)
clf = LogisticRegression(C=2, max_iter=2000, class_weight="balanced").fit(emb(tr_s), tr_y)
sb = clf.predict(emb(te_s))

f = LLMGapFilter(backend="api", model=WINNER, mode=MODE, client=CLIENT)
pos=[i for i,v in enumerate(sb) if v==1]; keep=[]
for k in range(0,len(pos),CHUNK): keep.extend(f.judge_batch([te_s[i] for i in pos[k:k+CHUNK]]))
sc=sb.copy()
for i,kp in zip(pos,keep):
    if not kp: sc[i]=0

def pm(y, t=0.207, seed=0):
    p=np.where(y==1)[0]; n=np.where(y==0)[0]; nn=int(round(len(p)*(1-t)/t))
    n=np.random.RandomState(seed).permutation(n)[:nn]; return np.sort(np.concatenate([p,n]))
idx=pm(te_y)
import pandas as pd
r=lambda nm,pred: dict(stage=nm, P=prf(te_y[idx],pred[idx])[0], R=prf(te_y[idx],pred[idx])[1], F1=prf(te_y[idx],pred[idx])[2])
out=pd.DataFrame([r("Stage B only",sb), r("+ Stage C (final)",sc),
                  {"stage":"Lan 2024 PubMedBERT","P":0.751,"R":0.907,"F1":0.821}]).set_index("stage")
print(f"prevalence-matched to {te_y[idx].mean():.1%}"); out

## Summary

`stage_c_filter(sentences)` drops the funnel's false positives (prior-work critiques, contribution claims, acknowledgments) with a cheap batched LLM call. On SAL, prevalence-matched, Stage B + Stage C approaches the Lan 2024 PubMedBERT SOTA (0.821) with a frozen general head + an LLM filter — and judges only ~6 sentences/paper, so it's ~cents per thousands of papers.